# TP3 · Recomendador personalizado de libros

Procesamiento del Lenguaje Natural · TUIA · UNR-FCEIA · Grupo 7

Integrantes: Civetta Valentino, Fucci Milagros, Winter Federico, Frank Maximiliano.

Este notebook sigue los ocho pasos del enunciado y mantiene sus títulos. Por ahora llega hasta el **Paso 3**. Los pasos 4 a 8 quedan como títulos vacíos para completar después.

| Paso | Qué hacemos | Quién lo corre |
|---|---|---|
| 1 | Cargar y revisar los libros | todos |
| 2 | Auditar el catálogo (detectores de contaminación) | todos |
| 3 | Registrar las preferencias del lector | **solo el lector elegido** |

## Preparación

Las librerías de las prácticas anteriores (pandas, scikit-learn, sentence-transformers) ya están en `requirements.txt`. Para este TP hacen falta además `langdetect`, `jellyfish` (Jaro-Winkler) y `spacy` con el modelo `es_core_news_lg`. En Colab hay que correr la celda de instalación una vez por sesión.

Las rutas de datos apuntan a `../data/`, igual que en el notebook del TP2 (se abre desde `src/`). En Colab hay que cambiar `RUTA` por la carpeta donde estén los archivos.

In [ ]:
# Solo en Colab: descomentar y correr una vez por sesion
# !pip install -q langdetect jellyfish spacy sentence-transformers
# !python -m spacy download es_core_news_lg

In [ ]:
import re
import os
import collections
import unicodedata

import numpy as np
import pandas as pd
import jellyfish
import spacy
from langdetect import detect_langs, DetectorFactory, LangDetectException
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, precision_recall_curve
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import NearestNeighbors

DetectorFactory.seed = 0      # langdetect es aleatorio: con la semilla fija repite el resultado
SEMILLA = 42
RUTA = "../data/"

# Paso 1. Cargar y revisar los libros

Cargamos `catalogo_tp3.jsonl` y nos quedamos con `slug` (identificador), `title`, `authors` y `synopsis`. Vamos anotando cuántos libros sale cada limpieza para poder informarlo al final.

In [ ]:
catalogo = pd.read_json(RUTA + "catalogo_tp3.jsonl", lines=True)
libros = catalogo[["slug", "title", "authors", "synopsis"]].copy()

# authors viene como lista: lo pasamos a texto ordenado para poder comparar autores
libros["autores"] = libros["authors"].apply(
    lambda a: ", ".join(sorted(a)) if isinstance(a, list) else str(a))
libros = libros.drop(columns="authors")

eliminados = {}
total_inicial = len(libros)
print("Libros en el catalogo:", total_inicial)

## 1.1 Sin sinopsis y slugs repetidos

In [ ]:
vacia = libros["synopsis"].isna() | (libros["synopsis"].astype(str).str.strip() == "")
eliminados["sin sinopsis"] = int(vacia.sum())
libros = libros[~vacia]

repetido = libros.duplicated("slug")
eliminados["slug repetido"] = int(repetido.sum())
libros = libros[~repetido]

print(eliminados)

## 1.2 Normalizar espacios

Las palabras en cursiva quedaron separadas por saltos de línea. Reemplazamos cada secuencia de espacios en blanco por un solo espacio y no tocamos ningún otro carácter, porque en el paso 2 algunos caracteres raros (invisibles, de otro alfabeto) van a ser evidencia. `\s` no incluye el espacio de ancho cero (U+200B), así que esa evidencia se conserva.

In [ ]:
con_salto = libros[libros["synopsis"].str.contains(r"\s{2,}|\n", regex=True)]
print("Sinopsis con saltos o espacios repetidos:", len(con_salto))
if len(con_salto):
    print("Antes:  ", repr(con_salto["synopsis"].iloc[0][:120]))

libros["synopsis"] = libros["synopsis"].apply(lambda t: re.sub(r"\s+", " ", t).strip())

if len(con_salto):
    print("Despues:", repr(libros.loc[con_salto.index[0], "synopsis"][:120]))

## 1.3 Sinopsis exactamente repetidas

Antes de borrar miramos qué hay detrás: cursos y series completas que repiten la misma sinopsis en todos sus volúmenes. Para esta práctica conservamos un libro por sinopsis (el primero que aparece).

In [ ]:
grupos = libros[libros.duplicated("synopsis", keep=False)].groupby("synopsis")
tamanos = grupos.size().sort_values(ascending=False)
print("Sinopsis que se repiten:", len(tamanos), "| libros involucrados:", int(tamanos.sum()))

mayor = tamanos.index[0]
print("\nEl grupo mas grande tiene", tamanos.iloc[0], "libros, por ejemplo:")
print(libros[libros["synopsis"] == mayor][["title", "autores"]].head(5).to_string(index=False))

repetida = libros.duplicated("synopsis", keep="first")
eliminados["sinopsis repetida"] = int(repetida.sum())
libros = libros[~repetida]
print("\nEliminados por sinopsis repetida:", eliminados["sinopsis repetida"])

## 1.4 Versiones casi iguales de una misma obra

Comparamos solo libros del mismo autor, para no cruzar los 8.000 títulos entre sí. Jaro-Winkler (`jellyfish`) mide la forma de los títulos, no el significado, así que «Conversaciones con Dios I» y «Conversaciones con Dios II» salen casi idénticos aunque sean libros distintos.

El criterio combina título y sinopsis. Dos libros del mismo autor se unen si pasan estas tres condiciones:

1. Los títulos «clave» tienen Jaro-Winkler de 0,95 o más. La clave es el título en minúsculas, sin acentos, sin lo que va entre paréntesis (traductor, edición, ilustrado) y sin números de tomo o volumen.
2. Los números de tomo o volumen son los mismos (arábigos, romanos en mayúscula o «primero», «segundo»…). Así «Epigramas I» y «Epigramas II» quedan separados.
3. La clave es idéntica, o las sinopsis comparten el 95 % o más de sus palabras (Jaccard sobre conjuntos de palabras). Un título casi igual con una sinopsis distinta, como «Mitología oriental» y «Mitología occidental», es otro libro.

De cada par que se une conservamos el libro de sinopsis más larga.

In [ ]:
UMBRAL_TITULO = 0.95
UMBRAL_SINOPSIS = 0.95
ORDINALES = r"\b(?:primero|segundo|tercero|cuarto|quinto)\b"

def sin_acentos(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))

def numeros_titulo(t):
    """Numeros de tomo o volumen: arabigos, romanos en mayuscula y ordinales."""
    return frozenset(re.findall(r"\d+|\b[IVXLC]+\b", t) + re.findall(ORDINALES, sin_acentos(t)))

def clave_titulo(t):
    t = re.sub(r"\(.*?\)", "", t)                      # traductor, edicion, ilustrado
    t = re.sub(r"\d+|\b[IVXLC]+\b", "", t)             # numeros de tomo
    t = sin_acentos(t)
    t = re.sub(ORDINALES + r"|\b(?:vol|tomo|libro|parte)\b", "", t)
    t = re.sub(r"[^a-z ]", "", t)
    return re.sub(r"\s+", " ", t).strip()

def jaccard(a, b):
    return len(a & b) / len(a | b)

libros = libros.assign(
    clave=libros["title"].apply(clave_titulo),
    numeros=libros["title"].apply(numeros_titulo),
    palabras=libros["synopsis"].apply(lambda t: set(re.findall(r"\w+", t.lower()))),
    largo=libros["synopsis"].str.len(),
).sort_values("largo", ascending=False)

conservados = {}        # autores -> libros que ya decidimos conservar
descartar, pares_unidos, pares_separados = [], [], []

for r in libros.itertuples():
    unido = False
    for q in conservados.get(r.autores, []):
        if jellyfish.jaro_winkler_similarity(r.clave, q.clave) < UMBRAL_TITULO:
            continue
        jac = jaccard(r.palabras, q.palabras)
        if r.numeros != q.numeros:
            pares_separados.append((r.title, q.title, round(jac, 2), "tomo o volumen distinto"))
        elif r.clave == q.clave or jac >= UMBRAL_SINOPSIS:
            pares_unidos.append((r.title, q.title, round(jac, 2)))
            unido = True
            break
        else:
            pares_separados.append((r.title, q.title, round(jac, 2), "sinopsis distinta"))
    if unido:
        descartar.append(r.slug)
    else:
        conservados.setdefault(r.autores, []).append(r)

eliminados["casi duplicado"] = len(descartar)
libros = libros[~libros["slug"].isin(descartar)].sort_index()
libros = libros.drop(columns=["clave", "numeros", "palabras", "largo"]).reset_index(drop=True)

print("Eliminados por casi duplicado:", len(descartar))
print("\nDos pares que decidimos unir:")
display(pd.DataFrame(pares_unidos, columns=["titulo A", "titulo B (conservado)", "jaccard"]).head(2))
print("Dos pares que decidimos separar:")
display(pd.DataFrame(pares_separados, columns=["titulo A", "titulo B", "jaccard", "motivo"])
        .groupby("motivo").head(1))

## 1.5 Resumen del paso 1

In [ ]:
print("Libros al empezar:       ", total_inicial)
for motivo, n in eliminados.items():
    print(f"  eliminados por {motivo}: {n}")
print("Libros disponibles:      ", len(libros))
assert total_inicial - sum(eliminados.values()) == len(libros)

print("\nCinco registros al azar:")
display(libros.sample(5, random_state=SEMILLA)[["slug", "title", "autores"]])

# Paso 2. Auditar el catálogo

Trabajamos con `muestra_etiquetada.csv` (500 libros, 100 contaminados). La dividimos **por libro** en 70 % para desarrollar los detectores y 30 % para medirlos. Las decisiones (umbrales, patrones) salen siempre de desarrollo y las cifras finales se informan sobre prueba.

Hay que recordar que la muestra está enriquecida: 20 % contaminado contra una proporción mucho menor en el catálogo completo, así que la precisión medida acá es más optimista que la que tendríamos sobre los 8.000 libros. Además el catálogo tiene mensajes que la muestra nunca mostró.

In [ ]:
muestra = pd.read_csv(RUTA + "muestra_etiquetada.csv", keep_default_na=False)

desarrollo, prueba = train_test_split(
    muestra, test_size=0.3, random_state=SEMILLA, stratify=muestra["contaminado"])
print(len(desarrollo), len(prueba))
print("contaminados en desarrollo:", int(desarrollo["contaminado"].sum()),
      "| en prueba:", int(prueba["contaminado"].sum()))

def medir(nombre, marcado, datos):
    y = datos["contaminado"].values
    marcado = np.asarray(marcado, dtype=bool)
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado)
    print(f"{nombre}: precision {p:.2f}, recall {r:.2f}, marcados {int(marcado.sum())}")
    return p, r

def oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]

# cada detector recibe un DataFrame con la columna synopsis y devuelve un arreglo de True/False
DETECTORES = {}

## 2.1 Expresiones regulares

Una expresión para URLs (con `http(s)://` o `www.`), otra para correos y otra para teléfonos con prefijo internacional. Las unimos en una sola.

In [ ]:
URL = r"https?://\S+|www\.\S+"
MAIL = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
TEL = r"\+\d[\d\s-]{8,}\d"
RE_BASE = re.compile("|".join([URL, MAIL, TEL]), re.IGNORECASE)

def det_regex(df):
    return df["synopsis"].apply(lambda t: bool(RE_BASE.search(t))).values

DETECTORES["regex"] = det_regex

marcado = det_regex(desarrollo)
medir("regex base (desarrollo)", marcado, desarrollo)

print("\nFalsos positivos (sinopsis legitimas marcadas):")
for t in desarrollo[marcado & (desarrollo["contaminado"] == 0)]["synopsis"]:
    print("  ", [m.group() for m in RE_BASE.finditer(t)])

perdidos = desarrollo[~marcado & (desarrollo["contaminado"] == 1)]
print(f"\nContaminados que la regex no ve: {len(perdidos)}. Algunos fragmentos:")
for f in perdidos["fragmento"].sample(min(8, len(perdidos)), random_state=SEMILLA):
    print("  ", ascii(f)[:110])

**Qué detecta bien y qué no.** La regex acierta con los enlaces escritos de forma explícita (`http://…`, `www.…`), los correos y los teléfonos con `+`. No ve los enlaces camuflados ni el texto promocional sin enlace. Los falsos positivos aparecen cuando una sinopsis real menciona un sitio que existe (en la muestra hay una que nombra `xkcd.com`), y una regex no sabe distinguir una mención legítima de un enlace insertado. Cada falso positivo es un libro legítimo que después no se puede recomendar.

## 2.2 Normalizar antes de buscar

Los enlaces disimulados usan palabras en lugar de signos («punto», «barra»), letras separadas por espacios, `[.]`, letras de otro alfabeto que se ven iguales a las latinas y caracteres invisibles. `normalizar(texto)` deshace cada truco y después volvemos a aplicar la regex.

Con `unicodedata` se puede ver qué es cada carácter: una `o` latina, una `о` cirílica y un espacio de ancho cero se ven casi igual (o directamente no se ven) pero tienen nombres y categorías distintas.

In [ ]:
for c in ("o", "о", "​", "⁠"):
    print(repr(c), unicodedata.name(c), unicodedata.category(c))

def con_invisibles(t):
    return any(unicodedata.category(c) == "Cf" for c in t)       # Cf = formato (invisibles)

inv = desarrollo["synopsis"].apply(con_invisibles)
print("\nSinopsis con caracteres invisibles -> contaminadas:",
      int((inv & (desarrollo["contaminado"] == 1)).sum()),
      "| legitimas:", int((inv & (desarrollo["contaminado"] == 0)).sum()))

Los invisibles también aparecen en sinopsis legítimas (guiones blandos, por ejemplo). Por sí solos no son evidencia: los quitamos para poder ver el texto que hay alrededor, no para marcar el libro.

In [ ]:
TLD = r"com|net|org|info|io|ar|example|test|invalid"
DOMINIO = r"\b[\w-]+(?:\.[\w-]+)*\.(?:%s)\b" % TLD
RE_AMPLIADA = re.compile("|".join([URL, MAIL, TEL, DOMINIO]), re.IGNORECASE)

# letras cirilicas y griegas que se ven como latinas -> su equivalente latino
HOMOGLIFOS = str.maketrans(
    "аеорсіухοАЕОРСІХ",
    "aeopciyxoAEOPCIX")

def normalizar(texto):
    t = unicodedata.normalize("NFKC", texto)                       # formas de ancho completo, ligaduras
    t = "".join(c for c in t if unicodedata.category(c) != "Cf")   # invisibles
    t = t.translate(HOMOGLIFOS)                                    # otro alfabeto
    t = re.sub(r"\s*(?:\[\.\]|\(\.\))\s*", ".", t)                 # tomoya [.] test
    t = re.sub(r"(?<!\w)(?:\S ){3,}\S(?!\w)",                      # h t t p s : / /  y  w w w .
               lambda m: m.group().replace(" ", ""), t)
    t = re.sub(r"(?i)\s+punto\s+(%s)\b(\s+barra\s+)?" % TLD,       # sitio punto test barra libro
               lambda m: "." + m.group(1) + ("/" if m.group(2) else ""), t)
    t = re.sub(r"(\w) \. (\w)", r"\1.\2", t)                       # novelalibre . example
    t = re.sub(r"(?i)\b(https?://|www\.)\s+", r"\1", t)
    return t

det_base_norm = lambda df: df["synopsis"].apply(lambda t: bool(RE_BASE.search(normalizar(t)))).values
det_norm = lambda df: df["synopsis"].apply(lambda t: bool(RE_AMPLIADA.search(normalizar(t)))).values
DETECTORES["normalizacion"] = det_norm

medir("regex base sobre texto crudo      ", det_regex(desarrollo), desarrollo)
medir("regex base sobre texto normalizado", det_base_norm(desarrollo), desarrollo)
medir("regex + dominios, normalizado     ", det_norm(desarrollo), desarrollo)

print("\nDos casos que la normalizacion recupero (fragmento original -> normalizado):")
rec = [f for f in desarrollo[desarrollo["contaminado"] == 1]["fragmento"]
       if not RE_BASE.search(f) and RE_AMPLIADA.search(normalizar(f)) and normalizar(f) != f]
for f in rec[:2]:
    print("  ", ascii(f)[:100]); print("  ->", normalizar(f)[:100])

La segunda regex agrega una sola cosa: los dominios sin `http` ni `www` (`algo.example`, `algo.test`, `algo.com`). Los falsos positivos de esta versión siguen siendo las menciones legítimas de sitios reales; se cuentan en la columna `falsos_positivos` de la tabla del paso 2.7.

## 2.3 Detectar el idioma de cada oración

Primero el idioma de la sinopsis completa con `langdetect`. Esperamos que un mensaje corto en inglés dentro de un texto largo en español no alcance a cambiar el resultado; la tabla lo confirma o lo desmiente.

In [ ]:
def idioma_completo(texto):
    try:
        return detect_langs(texto)[0].lang
    except LangDetectException:
        return "?"

des_idioma = desarrollo.assign(idioma=desarrollo["synopsis"].apply(idioma_completo))
display(pd.crosstab(des_idioma["contaminado"], des_idioma["idioma"]))

Dividimos cada sinopsis en oraciones y detectamos el idioma de cada una. Las oraciones cortas, las listas de nombres propios y las citas producen errores, así que hay que fijar dos valores: la cantidad mínima de palabras y la probabilidad mínima para creer que la oración **no** está en español. Probamos una grilla sobre desarrollo y elegimos la combinación de mayor recall que mantenga una precisión de al menos 0,90.

In [ ]:
def idiomas_oraciones(texto):
    salida = []
    for o in oraciones(texto):
        try:
            l = detect_langs(o)[0]
            salida.append((len(o.split()), l.lang, l.prob))
        except LangDetectException:
            pass
    return salida

desarrollo = desarrollo.assign(idiomas=desarrollo["synopsis"].apply(idiomas_oraciones))

def marca_idioma(idiomas, min_palabras, umbral):
    return any(n >= min_palabras and lang != "es" and p >= umbral for n, lang, p in idiomas)

filas = []
for mp in (2, 3, 4, 5, 6):
    for um in (0.7, 0.8, 0.9, 0.95, 0.99):
        pred = desarrollo["idiomas"].apply(lambda x: marca_idioma(x, mp, um))
        y = desarrollo["contaminado"].values
        filas.append(dict(min_palabras=mp, umbral=um,
                          precision=precision_score(y, pred, zero_division=0),
                          recall=recall_score(y, pred)))
grilla = pd.DataFrame(filas)
display(grilla.pivot(index="min_palabras", columns="umbral", values="precision").round(2))
display(grilla.pivot(index="min_palabras", columns="umbral", values="recall").round(2))

elegibles = grilla[grilla["precision"] >= 0.90]
if elegibles.empty:
    print("Ninguna combinacion llega a precision 0.90: usamos la de mejor precision")
    elegibles = grilla.sort_values("precision", ascending=False).head(1)
# ante empate de recall nos quedamos con la combinacion mas exigente (umbral y minimo mas altos)
mejor = elegibles.sort_values(["recall", "umbral", "min_palabras"], ascending=False).iloc[0]
MIN_PALABRAS, UMBRAL_IDIOMA = int(mejor["min_palabras"]), float(mejor["umbral"])
print(f"\nElegimos min_palabras={MIN_PALABRAS} y umbral={UMBRAL_IDIOMA} "
      f"(desarrollo: precision {mejor['precision']:.2f}, recall {mejor['recall']:.2f})")

def det_idioma(df):
    return df["synopsis"].apply(
        lambda t: marca_idioma(idiomas_oraciones(t), MIN_PALABRAS, UMBRAL_IDIOMA)).values

DETECTORES["idioma"] = det_idioma

Justificación de los dos valores (completar mirando las dos tablas de arriba): por qué subir el mínimo de palabras o el umbral baja el recall, y qué tipo de oración legítima (títulos en otro idioma, citas) hace caer la precisión cuando los valores son bajos.

## 2.4 Reconocer entidades

Aplicamos NER con spaCy (`es_core_news_lg`) a las oraciones de los fragmentos contaminados de desarrollo y a la misma cantidad de oraciones de sinopsis limpias.

In [ ]:
nlp = spacy.load("es_core_news_lg")

frases_malas = [o for f in desarrollo[desarrollo["contaminado"] == 1]["fragmento"] for o in oraciones(f)]
limpias = desarrollo[desarrollo["contaminado"] == 0]["synopsis"].apply(oraciones).explode().dropna()
frases_buenas = limpias.sample(len(frases_malas), random_state=SEMILLA).tolist()

def entidades(frases):
    cuenta, textos = collections.Counter(), collections.Counter()
    for doc in nlp.pipe(frases, batch_size=64):
        for e in doc.ents:
            cuenta[e.label_] += 1
            textos[(e.label_, e.text)] += 1
    return cuenta, textos

c_mal, t_mal = entidades(frases_malas)
c_bien, t_bien = entidades(frases_buenas)

print(len(frases_malas), "oraciones de cada grupo")
display(pd.DataFrame({"contaminadas": c_mal, "limpias": c_bien}).fillna(0).astype(int))
print("Entidades mas frecuentes en el texto insertado:", t_mal.most_common(8))
print("Entidades mas frecuentes en sinopsis limpias:  ", t_bien.most_common(8))

Respuesta (completar con la tabla): qué etiquetas aparecen en el texto insertado y no en una sinopsis normal, y si sirven para **detectar** o solo para **explicar** lo que ya encontraron los otros detectores (por ejemplo, qué tan seguido el NER etiqueta como organización o miscelánea un dominio que la regex ya había marcado).

## 2.5 Clasificar oraciones

El texto promocional sin enlaces solo se detecta aprendiendo de ejemplos. Armamos un conjunto de oraciones con los libros de desarrollo: las oraciones del fragmento son positivas y las oraciones de las sinopsis limpias son negativas. Entrenamos dos clasificadores como los del paso 5: uno con TF-IDF y otro con embeddings (`distiluse-base-multilingual-cased-v1`, el mismo modelo del TP2), los dos con `LogisticRegression(class_weight="balanced", max_iter=1000)`.

Un libro es sospechoso si alguna de sus oraciones supera el umbral. Elegimos ese umbral con `precision_recall_curve` para una precisión de 0,90, no con 0,5. Para que las probabilidades de desarrollo no sean optimistas (el modelo vería los mismos libros que se están midiendo) las calculamos con validación cruzada **por libro**: cada libro se puntúa con un modelo que no lo vio entrenar.

In [ ]:
modelo_emb = SentenceTransformer("distiluse-base-multilingual-cased-v1")

def armar_oraciones(df):
    filas = []
    for r in df.itertuples():
        textos = oraciones(r.fragmento) if r.contaminado == 1 else oraciones(r.synopsis)
        filas += [(r.slug, o, int(r.contaminado)) for o in textos]
    return pd.DataFrame(filas, columns=["slug", "oracion", "y"])

def ajustar(tipo, textos, y):
    """Entrena la regresion logistica y devuelve una funcion textos -> probabilidad de contaminado."""
    textos = list(textos)
    if tipo == "tfidf":
        vec = TfidfVectorizer()
        X = vec.fit_transform(textos)
        codificar = vec.transform
    else:
        codificar = lambda t: modelo_emb.encode(list(t), show_progress_bar=False)
        X = codificar(textos)
    clf = LogisticRegression(class_weight="balanced", max_iter=1000).fit(X, y)
    assert list(clf.classes_) == [0, 1]      # la columna 1 de predict_proba es "contaminado"
    return lambda t: clf.predict_proba(codificar(t))[:, 1]

def puntajes_libros(f_prob, df):
    """Puntaje de cada libro = mayor probabilidad entre sus oraciones."""
    por_libro = df["synopsis"].apply(oraciones).tolist()
    planas = [(i, o) for i, lista in enumerate(por_libro) for o in lista]
    puntaje = np.zeros(len(df))
    if planas:
        probs = f_prob([o for _, o in planas])
        for (i, _), p in zip(planas, probs):
            puntaje[i] = max(puntaje[i], p)
    return puntaje

def puntajes_cruzados(tipo, df, k=5):
    puntaje = np.zeros(len(df))
    for ent, val in StratifiedKFold(k, shuffle=True, random_state=SEMILLA).split(df, df["contaminado"]):
        o = armar_oraciones(df.iloc[ent])
        f = ajustar(tipo, o["oracion"], o["y"])
        puntaje[val] = puntajes_libros(f, df.iloc[val])
    return puntaje

def elegir_umbral(y, puntaje, precision_min=0.90):
    p, r, u = precision_recall_curve(y, puntaje)
    ok = np.where(p[:-1] >= precision_min)[0]
    if len(ok) == 0:
        print("Ninguno llega a la precision pedida: usamos el umbral de mayor precision")
        return u[np.argmax(p[:-1])]
    return u[ok[np.argmax(r[ok])]]

oraciones_des = armar_oraciones(desarrollo)
print("Oraciones de entrenamiento:", len(oraciones_des),
      "| positivas:", int(oraciones_des["y"].sum()))

y_des = desarrollo["contaminado"].values
clasificadores = {}
for tipo in ("tfidf", "emb"):
    puntaje_cv = puntajes_cruzados(tipo, desarrollo)
    umbral = elegir_umbral(y_des, puntaje_cv)
    f_prob = ajustar(tipo, oraciones_des["oracion"], oraciones_des["y"])    # modelo final, todo desarrollo
    clasificadores[tipo] = (f_prob, umbral)
    print(f"{tipo}: umbral elegido {umbral:.3f}")
    medir(f"{tipo} (desarrollo, validacion cruzada)", puntaje_cv >= umbral, desarrollo)

def hacer_detector(tipo):
    f_prob, umbral = clasificadores[tipo]
    return lambda df: puntajes_libros(f_prob, df) >= umbral

DETECTORES["clasificador_tfidf"] = hacer_detector("tfidf")
DETECTORES["clasificador_emb"] = hacer_detector("emb")

for nombre in ("clasificador_tfidf", "clasificador_emb"):
    medir(nombre + " (prueba)", DETECTORES[nombre](prueba), prueba)

Respuestas (completar mirando las salidas):

1. Por qué conviene clasificar oraciones y no la sinopsis completa (pista: de qué tamaño es el mensaje insertado frente al resto del texto).
2. Qué representación detectó mejor los libros contaminados de prueba, y a igual precisión cuál tuvo más recall.
3. El catálogo trae mensajes que la muestra no tiene. Qué representación debería generalizar mejor a frases nuevas con el mismo sentido, y cuál detecta mejor cadenas literales como una URL.

## 2.6 Encontrar libros copiados

Algunos libros no existen: copian la sinopsis de otro libro del catálogo y le agregan texto. Para cada libro buscamos su vecino más parecido con similitud del coseno (`NearestNeighbors`, `metric="cosine"`, sobre TF-IDF) en el catálogo que quedó del paso 1.

Una pareja con similitud alta y **autores distintos** es candidata, porque las series legítimas también comparten texto pero casi siempre tienen el mismo autor. Para decidir cuál de los dos es la copia usamos que la copia contiene a la otra sinopsis completa más texto agregado. Jaccard sobre palabras confirma el parecido.

In [ ]:
SIM_MIN = 0.70
JACCARD_MIN = 0.50

tfidf_nn = TfidfVectorizer()
X_nn = tfidf_nn.fit_transform(libros["synopsis"])
nn = NearestNeighbors(n_neighbors=3, metric="cosine", algorithm="brute").fit(X_nn)
dist, vecinos = nn.kneighbors(X_nn)

pares = {}
for i in range(len(libros)):
    for d, j in zip(dist[i], vecinos[i]):
        if j != i:                                  # el primer vecino distinto de si mismo
            pares[tuple(sorted((i, j)))] = 1 - d
            break

filas, copias = [], set()
for (i, j), sim in pares.items():
    a, b = libros.iloc[i], libros.iloc[j]
    if sim < SIM_MIN or a["autores"] == b["autores"]:
        continue
    jac = jaccard(set(re.findall(r"\w+", a["synopsis"].lower())),
                  set(re.findall(r"\w+", b["synopsis"].lower())))
    corta, larga = (a, b) if len(a["synopsis"]) <= len(b["synopsis"]) else (b, a)
    contiene = corta["synopsis"] in larga["synopsis"]
    if contiene and jac >= JACCARD_MIN:
        copias.add(larga["slug"])
    filas.append(dict(copia_probable=larga["title"], autor_copia=larga["autores"],
                      original=corta["title"], autor_original=corta["autores"],
                      similitud=round(sim, 2), jaccard=round(jac, 2), contiene=contiene))

candidatos = pd.DataFrame(filas)
print("Parejas con similitud >=", SIM_MIN, "y autores distintos:", len(candidatos))
print("Libros marcados como copia (contiene a la otra sinopsis y jaccard >=", JACCARD_MIN, "):", len(copias))
if len(candidatos):
    display(candidatos.sort_values("similitud", ascending=False).head(10))
    print("Parejas que quedaron para revisar a mano (no se contienen):")
    display(candidatos[~candidatos["contiene"]].head(5))

DETECTORES["copia"] = lambda df: df["slug"].isin(copias).values
medir("copia (desarrollo)", DETECTORES["copia"](desarrollo), desarrollo)
medir("copia (prueba)", DETECTORES["copia"](prueba), prueba)

Si hay parejas que no se contienen pero tienen similitud alta, hay que mirarlas a mano: el texto agregado puede estar en el medio o la copia puede haber retocado la sinopsis original. Esas parejas no se marcan automáticamente.

## 2.7 Combinar y entregar

Un libro es sospechoso si al menos un detector lo marca. Marcar todo tampoco sirve: cada libro legítimo que se elimina es uno que nunca se va a poder recomendar, así que miramos la precisión y el recall de cada detector y de la unión sobre la parte de **prueba**.

In [ ]:
y_pru = prueba["contaminado"].values
marcas_pru = {nombre: np.asarray(fn(prueba), dtype=bool) for nombre, fn in DETECTORES.items()}
marcas_pru["COMBINADO (al menos uno)"] = np.any(list(marcas_pru.values()), axis=0)

tabla = pd.DataFrame([
    dict(detector=n,
         precision=precision_score(y_pru, m, zero_division=0),
         recall=recall_score(y_pru, m),
         marcados=int(m.sum()),
         falsos_positivos=int((m & (y_pru == 0)).sum()))
    for n, m in marcas_pru.items()]).round(2)
display(tabla)

Ahora aplicamos todos los detectores al catálogo completo del paso 1 y guardamos `grupo7_sospechosos.csv` (columnas `slug` y `motivo`). Un libro que marcan varios detectores lleva varios motivos unidos con `+`. Si la regex ampliada solo marca lo que la regex base no veía, el motivo es `normalizacion`, y los dos clasificadores se agrupan en `clasificador`.

Con el catálogo de unos 8.500 libros el clasificador de embeddings codifica decenas de miles de oraciones: en Colab con GPU tarda unos minutos y con CPU bastante más.

In [ ]:
marcas = {nombre: np.asarray(fn(libros), dtype=bool) for nombre, fn in DETECTORES.items()}

motivos_libro = {
    "regex": marcas["regex"],
    "normalizacion": marcas["normalizacion"] & ~marcas["regex"],
    "idioma": marcas["idioma"],
    "clasificador": marcas["clasificador_tfidf"] | marcas["clasificador_emb"],
    "copia": marcas["copia"],
}

def juntar_motivos(i):
    return "+".join(m for m, v in motivos_libro.items() if v[i])

sospechoso = np.any(list(motivos_libro.values()), axis=0)
sospechosos = pd.DataFrame({
    "slug": libros.loc[sospechoso, "slug"].values,
    "motivo": [juntar_motivos(i) for i in np.where(sospechoso)[0]]})

# si ya se corrio el paso 3, el lector agrego libros con motivo revision_manual: no los perdemos
guardar = sospechosos
if os.path.exists(RUTA + "grupo7_sospechosos.csv"):
    previos = pd.read_csv(RUTA + "grupo7_sospechosos.csv")
    manuales = previos[(previos["motivo"] == "revision_manual") & ~previos["slug"].isin(sospechosos["slug"])]
    guardar = pd.concat([sospechosos, manuales], ignore_index=True)
guardar.to_csv(RUTA + "grupo7_sospechosos.csv", index=False)

catalogo_limpio = libros[~sospechoso].reset_index(drop=True)
catalogo_limpio.to_csv(RUTA + "catalogo_limpio.csv", index=False)

print("Libros revisados:", len(libros))
print("Sospechosos:", len(sospechosos), f"({len(sospechosos) / len(libros):.1%})")
print("Catalogo limpio:", len(catalogo_limpio))
display(sospechosos["motivo"].value_counts())

# Paso 3. Registrar las preferencias de un lector

**Esta parte la corre solo la persona del grupo que elegimos como lector de referencia**, en casa y antes de ver ninguna predicción. Elegido el lector: ______________________ .

Mezclamos el catálogo limpio una sola vez con `df.sample(frac=1, random_state=42)`. Al lector se le muestra el título y la sinopsis de cada libro, en ese orden, y responde «¿Me interesaría leer este libro?»:

| Tecla | Respuesta | `interes` |
|---|---|---|
| `s` | Sí, me interesaría leerlo | 1 |
| `n` | No me interesaría leerlo | 0 |
| `v` | No puedo decidir | vacío |
| `m` | Parece contaminado y el paso 2 no lo detectó | vacío (y va a sospechosos con motivo `revision_manual`) |
| `q` | Corto acá y sigo después | no se guarda nada |

Se sigue hasta reunir al menos 100 respuestas válidas y 20 de cada clase. Cada respuesta se guarda en el archivo en el momento (igual que el scraper del TP1 escribe libro por libro), así que se puede cortar y retomar sin perder nada: al volver a correr la celda sigue desde el primer libro sin responder.

In [ ]:
from IPython.display import clear_output

ARCH_PREF = RUTA + "grupo7_preferencias.csv"
ARCH_SOSP = RUTA + "grupo7_sospechosos.csv"
MINIMO_VALIDAS, MINIMO_POR_CLASE = 100, 20

limpio = pd.read_csv(RUTA + "catalogo_limpio.csv", keep_default_na=False)
orden = limpio.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

if os.path.exists(ARCH_PREF):
    prefs = pd.read_csv(ARCH_PREF, keep_default_na=False)       # el interes vacio queda como ""
else:
    prefs = pd.DataFrame(columns=["slug", "title", "synopsis", "interes"])

def conteo(p):
    v = pd.to_numeric(p["interes"], errors="coerce")
    return int((v == 1).sum()), int((v == 0).sum())

def completo(p):
    pos, neg = conteo(p)
    return pos + neg >= MINIMO_VALIDAS and pos >= MINIMO_POR_CLASE and neg >= MINIMO_POR_CLASE

def marcar_revision_manual(slug):
    sos = pd.read_csv(ARCH_SOSP) if os.path.exists(ARCH_SOSP) else pd.DataFrame(columns=["slug", "motivo"])
    if slug not in set(sos["slug"]):
        sos.loc[len(sos)] = [slug, "revision_manual"]
        sos.to_csv(ARCH_SOSP, index=False)

RESPUESTAS = {"s": 1, "n": 0, "v": "", "m": ""}
vistos = set(prefs["slug"])

for libro in orden.itertuples():
    if completo(prefs):
        break
    if libro.slug in vistos:
        continue
    pos, neg = conteo(prefs)
    clear_output(wait=True)
    print(f"Respuestas validas: {pos + neg}  (si: {pos}, no: {neg})   meta: {MINIMO_VALIDAS} y {MINIMO_POR_CLASE} de cada clase\n")
    print(libro.title, "\n")
    print(libro.synopsis, "\n")
    r = ""
    while r not in RESPUESTAS and r != "q":
        r = input("Te interesaria leerlo? [s / n / v = no puedo decidir / m = contaminado / q = salir]: ").strip().lower()
    if r == "q":
        break
    if r == "m":
        marcar_revision_manual(libro.slug)
    prefs.loc[len(prefs)] = [libro.slug, libro.title, libro.synopsis, RESPUESTAS[r]]
    prefs.to_csv(ARCH_PREF, index=False)
    vistos.add(libro.slug)

clear_output(wait=True)
print("Listo." if completo(prefs) else "Quedaron respuestas pendientes: volver a correr esta celda.")

## Resumen de las preferencias

Un libro sin valorar tiene preferencia desconocida: queda con `interes` vacío y no se convierte en 0.

In [ ]:
prefs = pd.read_csv(ARCH_PREF)               # aca el interes vacio pasa a NaN
print("Libros vistos:", len(prefs))
print(prefs["interes"].map({1: "interes=1", 0: "interes=0"}).fillna("sin valorar").value_counts())

pos, neg = conteo(prefs.fillna(""))
print(f"\nRespuestas validas: {pos + neg} (interes=1: {pos}, interes=0: {neg})")
assert completo(prefs.fillna("")), "Todavia faltan respuestas para llegar a 100 validas y 20 de cada clase"

# Paso 4. Separar los libros para aprender y evaluar

Pendiente.

# Paso 5. Entrenar dos clasificadores

Pendiente.

# Paso 6. Comprobar las predicciones

Pendiente.

# Paso 7. Convertir las predicciones en recomendaciones

Pendiente.

# Paso 8. Recomendar del catálogo completo

Pendiente.